<a href="https://colab.research.google.com/github/tavish-1721/Tavish-/blob/main/CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import torch
import torch.nn as nn #a collection of ready-made building blocks for building neural networks.

image=torch.randn(1,1,28,28) #channel=1(black and white),fill 28*28 with random numbers

conv=nn.Conv2d(
    in_channels=1,
    out_channels=8,
    kernel_size=3,
    stride=1,
    padding=0
)

output=conv(image)
print("Input Image",image.shape)
print("Output Image",output.shape)

Input Image torch.Size([1, 1, 28, 28])
Output Image torch.Size([1, 8, 26, 26])


In [7]:
import torch
import torch.nn as nn

x=torch.randn(1,8,28,28)

pool=nn.MaxPool2d(kernel_size=2)
y=pool(x)

print("Before Pooling",x.shape)
print("After Pooling",y.shape)
#important information is kept at important info like channels are same but removed useless part of height and weight
#pouling keeps strong edges,clear shape,dominant pattern.

Before Pooling torch.Size([1, 8, 28, 28])
After Pooling torch.Size([1, 8, 14, 14])


In [64]:

import torch #main pytorch library
import torch.nn as nn #neural network layer
import torch.optim as optim #optimizers
from torchvision.datasets import MNIST #MNIST databse
from torchvision import transforms #image transformations
from torch.utils.data import DataLoader #batch loading

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:",device)

transform=transforms.ToTensor() #downloads 70,000 handwritten digit images, converts pixels from 0-255 to 0.0-1.0

train_data=MNIST( #train_data → 60,000 images  (model learns from these)
    root='data',#where to save the downloaded dataset on your computer,creates a folder called "data"
    train=True,#gives 60,000 training images
    download=True,#True  → download from internet if not already on disk
    transform=transform#converts raw image → PyTorch tensor
)
test_data=MNIST( #test_data  → 10,000 images  (model tested on these)
    root='data',
    train=False,#train=False  # gives 10,000 testing images
    download=True,
    transform=transform
)
#feeds images in groups of 64 at a time:
train_loader=DataLoader(train_data,batch_size=64,shuffle=True) #shuffle=True  → random order each epoch,for learning otherwise it will remember the pattern instead of learning.
test_loader=DataLoader(test_data,batch_size=64,shuffle=False)#shuffle=False → fixed order for testing

class SimpleCNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.features=nn.Sequential(#self.features is a container that holds multiple layers in order using nn.Sequential
        nn.Conv2d(1,8,kernel_size=3,padding=1),#8 kernels detect patterns
        nn.ReLU(),#remove negatives
        nn.MaxPool2d(2),# halve size
        nn.Conv2d(8,16,kernel_size=3,padding=1),# 16 kernels detect deeper patterns
        nn.ReLU(),
        nn.MaxPool2d(2)#halve size again
    )
   #flatten- 16×7×7=784
    self.classifier=nn.Sequential(#self.classifier is the decision making part of your model
        nn.Linear(16*7*7,64),#compress 784 features to 64 features
        nn.ReLU(),#removes all negative values,keeps only positive signals,adds non-linearity so model can learn complex patterns
        nn.Linear(64,10)# 10 scores (one per digit), (0,1,2,3,4,5,6,7,8,9),thinks: "which digit does this look like?" from 64 inputs ,it will deliver 10 outputs
    )

  def forward(self,x):
      x=self.features(x) #run through conv layers
      x=x.view(x.size(0),-1)#flatten [64,16,7,7] → [64,784]
      x=self.classifier(x)# run through linear layers
      return x

model=SimpleCNN().to(device)  # create model

loss_fn=nn.CrossEntropyLoss()# measures how wrong predictions are

optimizer=optim.Adam(model.parameters(),lr=0.001)# updates weights

epochs=3# # repeat training 3 times

for epoch in range(epochs):
   model.train() #switching to training mode
   total_loss=0 #reset loss counter
   for images,labels in train_loader: #937 batches
      images=images.to(device)
      labels=labels.to(device)

      optimizer.zero_grad()# clear old gradients
      outputs=model(images)     # forward pass,with forward pass → model takes input and produces output
      loss=loss_fn(outputs,labels)# measure error
      loss.backward()# compute gradients
      optimizer.step()  # update weights

      total_loss+=loss.item()  # track total loss

   print(f"Epoch{epoch+1}/{epochs}|Loss:{total_loss:.4f}")

   model.eval()#switches off training behaviour
   correct=0
   total=0

   with torch.no_grad():#no gradient tracking needed
    for images,labels in test_loader:
      images=images.to(device)
      labels=labels.to(device)

      output=model(images)  # forward pass,with forward pass → model takes input and produces output
      predictions=output.argmax(dim=1)#picks digit with highest score

      correct+=(predictions==labels).sum().item()#percentage of correct predictions
      total+=labels.size(0)


    accuracy=100*correct/total
    print(f"Test accuracy:{accuracy:.2f}%")

   torch.save(model.state_dict(),"simple_cnn_mnist.pth")
   print("Model saved")






using device: cpu
Epoch1/3|Loss:313.1531
Test accuracy:96.89%
Model saved
Epoch2/3|Loss:89.3952
Test accuracy:97.55%
Model saved
Epoch3/3|Loss:64.8107
Test accuracy:98.40%
Model saved
